# Rephrase

As mentioned by the TA, the best way for us to improve the amount of far left samples (the lowest represented in our current dataset) is to have an LLM rephrase what we currently have.

According to the Synthetic Data playbook released by HuggingFace, the best model to do this is SmolLM-2 (of the models they tested). Having a larger model for this doesn't help, and we just need a good prompt.

# Imports

In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import polars as pl
import re

# Rephrasing

In [2]:
checkpoint = "HuggingFaceTB/SmolLM2-1.7B-Instruct"

device = "cuda" # for GPU usage or "cpu" for CPU usage
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
# for multiple GPUs install accelerate and do `model = AutoModelForCausalLM.from_pretrained(checkpoint, device_map="auto")`
model = AutoModelForCausalLM.from_pretrained(checkpoint).to(device)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:86: UserWarning: 
Access to the secret `HF_TOKEN` has not been granted on this notebook.
You will not be requested again.
Please restart the session if you want to be prompted again.
  warnings.warn(


config.json:   0%|          | 0.00/908 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

In [9]:
system_prompt = """
Rewrite the provided headline and provided article summary, while maintaining the same political ideology expressed in them.

IMPORTANT: The emotion and political leaning of the headline and article summary should be the same.

IMPORTANT: You **must** provide a rewritten headline and article summary. Make sure you are not repeating the provided headline and summary or leaving your responses
blank. If you are unable to rewrite the headline and/or article summary, you can paraphrase or rephrase them while maintaining the same politicial ideology.
Do **not** include the name of the news source in your headline or summary.

Output only the headline and article summary, nothing else.
"""

In [12]:
def generate_synthetic_examples(file_name: str, num_rewrites_per_example: int, output_file_name: str):

  # Load examplesfrom csv
  examples = pl.read_csv(file_name)

  # Conduct inference with examples
  rephrase_df = pl.DataFrame(schema=
        {
            "outlet": pl.String(),
            "bias": pl.String(),
            "headline": pl.String(),
            "summary": pl.String(),
            "ground_news_interest_url": pl.String()
        })

  for _ in range(num_rewrites_per_example):
    for i in range(len(examples)):
      headline = examples[i, "headline"]
      summary = examples[i, "summary"]

      messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": f"Headline: {headline}\nSummary: {summary}"}]
      input_text = tokenizer.apply_chat_template(messages, tokenize=False)
      inputs = tokenizer.encode(input_text, return_tensors="pt").to(device)
      outputs = model.generate(inputs, max_new_tokens=400, temperature=0.2, top_p=0.9, do_sample=True)

      decoded_output = tokenizer.decode(outputs[0], skip_special_tokens=False)

      # Extract the assistant's response portion first
      # The text format of the LLM's output <im_start>assistant text <im_end>
      assistant_part = decoded_output.split("<|im_start|>assistant")[-1].split("<|im_end|>")[0]

      # Regex to find Headline and Summary content
      # matches "Headline: ..." and "Summary: ..." and captures the text group
      h_match = re.search(r"Headline:\s*(.*?)(?=\nSummary:|$)", assistant_part, re.DOTALL | re.IGNORECASE)
      s_match = re.search(r"Summary:\s*(.*)", assistant_part, re.DOTALL | re.IGNORECASE)

      generated_headline = h_match.group(1).strip() if h_match else ""
      generated_summary = s_match.group(1).strip() if s_match else ""

      rephrase_df = pl.concat([rephrase_df, pl.DataFrame({
              "outlet": [examples[i, "outlet"]],
              "bias": [examples[i, "bias"]],
              "headline": [generated_headline],
              "summary": [generated_summary],
              "ground_news_interest_url": [examples[i, "ground_news_interest_url"]],
          })], how = "vertical")

  # Save output
  rephrase_df.write_csv(output_file_name)

In [ ]:
generate_synthetic_examples("far_left_total.csv", 5, "far_left_synthetic.csv")

In [ ]:
generate_synthetic_examples("left_total.csv", 3, "left_synthetic.csv")

In [13]:
generate_synthetic_examples("far_right_total.csv", 5, "far_right_synthetic.csv")